# Floquet Model

The Floquet model describes a periodically driven two-qubit quantum system.
Unlike a system governed by a time-independent Hamiltonian, the dynamics are
generated by a sequence of interactions that repeat over a fixed period $T$.

The system evolves according to the Schrödinger equation:

$$
i\frac{d}{dt}|\psi(t)\rangle = H(t)|\psi(t)\rangle
$$

where the Hamiltonian is periodic:

$$
H(t+T)=H(t)
$$

During one driving period, different interaction terms are applied
sequentially. The Hamiltonian is defined as:

$$
H(t)=
\begin{cases}
H_1 = J_{xx}X\otimes X, & 0<t<T_1 \\
H_2 = hZ, & T_1<t<T_2 \\
H_3 = J_{yy}Y\otimes Y, & T_2<t<T
\end{cases}
$$

The $XX$ and $YY$ terms describe interactions between the two qubits, while
the $Z$ term represents a local field. Since these interaction terms can be
non-commuting, the order in which they are applied affects the quantum
evolution.

The complete sequence defines one Floquet period. By repeatedly applying this
period, the system can be studied over multiple driving cycles and can exhibit
non-trivial periodically driven quantum behavior.

### PAULI-BASIS MEASUREMENTS for Floquet Model

In [ ]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService(channel="ibm_quantum_platform")
backend = service.backend("ibm_fez")

shots = # type in the number of shots depending up on your problem

iterations = # number of Floquet steps

# FLOQUET parameters (plain variables, like LMG code)
alpha =
beta  =
gamma =

print("Backend:", backend.name)
print("Floquet steps:", iterations)

# FLOQUET 2-QUBIT MODEL CIRCUIT
def floquet_circuit(n, alpha, beta, gamma):
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.h(1)

    for _ in range(n):

        # exp(-i alpha XX)
        qc.h(0)
        qc.h(1)
        qc.cx(0, 1)
        qc.rz(2 * alpha, 1)
        qc.cx(0, 1)
        qc.h(0)
        qc.h(1)

        # exp(-i beta Z)
        qc.rz(2 * beta, 0)

        # exp(-i gamma YY)
        qc.sdg(0)
        qc.sdg(1)
        qc.h(0)
        qc.h(1)
        qc.cx(0, 1)
        qc.rz(2 * gamma, 1)
        qc.cx(0, 1)
        qc.h(0)
        qc.h(1)
        qc.s(0)
        qc.s(1)

    return qc

# BASIS ROTATIONS + MEASUREMENT
def add_basis_and_measure(qc, basis):
    qc2 = qc.copy()
    for q, b in enumerate(basis):
        if b == "X":
            qc2.h(q)
        elif b == "Y":
            qc2.sdg(q)
            qc2.h(q)
    qc2.measure_all()
    return qc2

bases = ["ZZ","ZX","ZY","ZI",
         "XZ","XX","XY","XI",
         "YZ","YX","YY","YI",
         "IZ","IX","IY"]

# BUILD CIRCUITS (FLOQUET STEP SWEEP)
all_circuits = []

for n in range(1, iterations + 1):
    for basis in bases:
        base = floquet_circuit(n, alpha, beta, gamma)
        full = add_basis_and_measure(base, basis)
        all_circuits.append(full)

print("Total circuits submitted:", len(all_circuits))  # 75 Circuits

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuits = pm.run(all_circuits)

sampler = Sampler(mode=backend)
job = sampler.run(isa_circuits, shots=shots)

print("\nSAVE THIS JOB ID:")
print(job.job_id())

## Retrieving the Job Results and Performing State Analysis

After submitting the quantum circuits to the IBM Quantum backend, the Job ID
printed by the execution code should be saved. The Job ID uniquely identifies
the submitted experiment and allows the experimental results to be retrieved
later without executing the quantum circuits again.

The saved Job ID is entered into the following analysis code:

```python
job = service.job("YOUR_JOB_ID")